# ML · Entrenamiento y Registro de Modelo — Riesgo de Retraso en Entrega

## Propósito

Entrenar un modelo de clasificación (Random Forest) para predecir el riesgo de retraso
en la entrega de pedidos, usando las features construidas en `ml_feature_engineering`.
El modelo entrenado se registra en el Unity Catalog Model Registry de Databricks.

## Fuente

- `gld_dataco.supply_chain.desc_ml_features_orders` — tabla de features (salida del paso anterior)

## Features de entrada

| Feature | Tipo | Preprocesamiento |
|---|---|---|
| `shipping_mode` | Categórica | StringIndexer → OHE |
| `days_for_shipment_scheduled` | Numérica | Passthrough |
| `market` | Categórica | StringIndexer → OHE |
| `order_region` | Categórica | StringIndexer → OHE |
| `es_sospechoso_fraude` | Binaria (int) | Passthrough |
| `mes_pedido` | Numérica | Passthrough |
| `dia_semana_pedido` | Numérica | Passthrough |

## Variable objetivo

- `late_delivery_risk` — 1 = retraso, 0 = a tiempo

## Algoritmo

- **RandomForestClassifier** (PySpark MLlib)
  - `numTrees=100`, `maxDepth=10`, `seed=42`
  - Sin necesidad de escalar features (árbol-based)

## Registro

- **MLflow Experiment:** `dataco_late_delivery_risk`
- **Modelo registrado:** `gld_dataco.supply_chain.ml_late_delivery_risk`
- **Alias:** `champion` → versión que entra a producción

## Métricas registradas

- `accuracy`, `f1`, `precision`, `recall` (clase 1 — retraso)
- `feature_importances` como JSON (para interpretabilidad)

## Notas

- Split 80/20 estratificado por `late_delivery_risk`
- El modelo anterior pierde el alias `champion` automáticamente al asignarlo a la nueva versión
- Este notebook es idempotente: cada ejecución crea una nueva versión del modelo

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("gold_catalog",     "gld_dataco",    "Catálogo Gold")
dbutils.widgets.text("gold_schema",      "supply_chain",  "Esquema Gold")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema",  "supply_chain",  "Esquema Metadata")
dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo si no corre dentro del Workflow)")

gold_catalog     = dbutils.widgets.get("gold_catalog")
gold_schema      = dbutils.widgets.get("gold_schema")
metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema  = dbutils.widgets.get("metadata_schema")

GOLD_ML_FEATURES_FQN = f"{gold_catalog}.{gold_schema}.desc_ml_features_orders"
MODEL_NAME_UC        = f"{gold_catalog}.{gold_schema}.ml_late_delivery_risk"
EXPERIMENT_NAME      = "/dataco_late_delivery_risk"
TABLA_ETL_LOG        = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"

print(f"Tabla de features : {GOLD_ML_FEATURES_FQN}")
print(f"Modelo UC         : {MODEL_NAME_UC}")
print(f"Experimento MLflow: {EXPERIMENT_NAME}")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 2: Inicio de log de ejecución
# ============================================================

import uuid

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID   = str(uuid.uuid4())
NOTEBOOK = "ml_train_register"
CAPA     = "gold"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("Corrida manual: log_id no se publica como Task Value")

print(f"Registro EN_PROCESO creado: {LOG_ID}")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 3: Cargar features y preparar datos
# ============================================================

from pyspark.sql import functions as F

df_features = (
    spark.table(GOLD_ML_FEATURES_FQN)
    .select(
        "shipping_mode",
        "days_for_shipment_scheduled",
        "market",
        "order_region",
        "es_sospechoso_fraude",
        "mes_pedido",
        "dia_semana_pedido",
        "late_delivery_risk"
    )
    .dropna()
)

total         = df_features.count()
positivos     = df_features.filter(F.col("late_delivery_risk") == 1).count()
negativos     = total - positivos
pct_positivos = round(positivos / total * 100, 1)

print(f"Total de registros limpios  : {total}")
print(f"Retrasos  (clase 1)         : {positivos} ({pct_positivos}%)")
print(f"A tiempo  (clase 0)         : {negativos} ({round(100-pct_positivos, 1)}%)")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 4: Pipeline de preprocesamiento
# ============================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

# --- Indexers para columnas categóricas ---
idx_shipping = StringIndexer(
    inputCol="shipping_mode", outputCol="shipping_mode_idx", handleInvalid="keep"
)
idx_market = StringIndexer(
    inputCol="market", outputCol="market_idx", handleInvalid="keep"
)
idx_region = StringIndexer(
    inputCol="order_region", outputCol="order_region_idx", handleInvalid="keep"
)

# --- One-Hot Encoding ---
ohe = OneHotEncoder(
    inputCols=["shipping_mode_idx", "market_idx", "order_region_idx"],
    outputCols=["shipping_mode_ohe", "market_ohe", "order_region_ohe"],
    handleInvalid="keep"
)

# --- Ensamble del vector de features ---
assembler = VectorAssembler(
    inputCols=[
        "shipping_mode_ohe",
        "days_for_shipment_scheduled",
        "market_ohe",
        "order_region_ohe",
        "es_sospechoso_fraude",
        "mes_pedido",
        "dia_semana_pedido"
    ],
    outputCol="features",
    handleInvalid="skip"
)

# --- Clasificador Random Forest ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="late_delivery_risk",
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline = Pipeline(stages=[idx_shipping, idx_market, idx_region, ohe, assembler, rf])

print("Pipeline definido: StringIndexer x3 -> OHE -> VectorAssembler -> RandomForest")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 5: Split y entrenamiento
# ============================================================

# Split estratificado 80/20
df_train, df_test = df_features.randomSplit([0.8, 0.2], seed=42)

print(f"Registros de entrenamiento : {df_train.count()}")
print(f"Registros de test          : {df_test.count()}")
print("Entrenando modelo...")

modelo = pipeline.fit(df_train)

print("Entrenamiento completado")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 6: Evaluación del modelo
# ============================================================

import json
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

df_pred = modelo.transform(df_test)

# Evaluadores
eval_acc = MulticlassClassificationEvaluator(
    labelCol="late_delivery_risk", predictionCol="prediction", metricName="accuracy"
)
eval_f1 = MulticlassClassificationEvaluator(
    labelCol="late_delivery_risk", predictionCol="prediction", metricName="f1"
)
eval_prec = MulticlassClassificationEvaluator(
    labelCol="late_delivery_risk", predictionCol="prediction",
    metricName="precisionByLabel", metricLabel=1.0
)
eval_rec = MulticlassClassificationEvaluator(
    labelCol="late_delivery_risk", predictionCol="prediction",
    metricName="recallByLabel", metricLabel=1.0
)
eval_auc = BinaryClassificationEvaluator(
    labelCol="late_delivery_risk", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)

accuracy  = eval_acc.evaluate(df_pred)
f1        = eval_f1.evaluate(df_pred)
precision = eval_prec.evaluate(df_pred)
recall    = eval_rec.evaluate(df_pred)
auc_roc   = eval_auc.evaluate(df_pred)

# Feature importances (top 10)
rf_model         = modelo.stages[-1]
feature_names    = assembler.getInputCols()
importances_list = [
    {"feature": name, "importance": round(float(imp), 4)}
    for name, imp in sorted(
        zip(feature_names, rf_model.featureImportances),
        key=lambda x: x[1], reverse=True
    )[:10]
]
feature_importances_json = json.dumps(importances_list)

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1        : {f1:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"AUC-ROC   : {auc_roc:.4f}")
print(f"Top features: {feature_importances_json}")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 7: Registrar en MLflow + Unity Catalog
# ============================================================

import mlflow
import mlflow.spark

# Unity Catalog como registry
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=f"rf_dataco_{id_ejecucion}") as run:

    # Parámetros del modelo
    mlflow.log_param("algoritmo",   "RandomForestClassifier")
    mlflow.log_param("num_trees",   100)
    mlflow.log_param("max_depth",   10)
    mlflow.log_param("seed",        42)
    mlflow.log_param("train_size",  df_train.count())
    mlflow.log_param("test_size",   df_test.count())

    # Métricas
    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("f1",        f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)
    mlflow.log_metric("auc_roc",   auc_roc)

    # Feature importances como artefacto
    mlflow.log_text(feature_importances_json, "feature_importances.json")

    # Loguear y registrar el modelo en Unity Catalog
    model_info = mlflow.spark.log_model(
        spark_model=modelo,
        artifact_path="model",
        registered_model_name=MODEL_NAME_UC
    )

    run_id = run.info.run_id

print(f"Modelo registrado: {MODEL_NAME_UC}")
print(f"Run ID           : {run_id}")
print(f"Model URI        : {model_info.model_uri}")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 8: Asignar alias 'champion'
# ============================================================

from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")

# Obtener la última versión recién registrada
versions = client.search_model_versions(f"name='{MODEL_NAME_UC}'")
ultima_version = max(versions, key=lambda v: int(v.version))
numero_version = ultima_version.version

# Asignar alias 'champion' (reemplaza al anterior automáticamente)
client.set_registered_model_alias(
    name=MODEL_NAME_UC,
    alias="champion",
    version=numero_version
)

# Publicar versión como Task Value para el notebook de predicción
try:
    dbutils.jobs.taskValues.set(key="model_version", value=numero_version)
    print("model_version publicado como Task Value")
except Exception:
    print("Corrida manual: model_version no se publica como Task Value")

print(f"Alias 'champion' asignado a la version {numero_version} de {MODEL_NAME_UC}")

In [ ]:
# ============================================================
# ML_TRAIN_REGISTER — Celda 9: Cerrar log de ejecución
# ============================================================

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado           = 'EXITOSO',
    fecha_fin        = current_timestamp(),
    filas_procesadas = {df_test.count()}
WHERE log_id = '{LOG_ID}'
""")

print(f"etl_log actualizado: EXITOSO")
print(f"Modelo champion    : {MODEL_NAME_UC} v{numero_version}")
print(f"Run ID MLflow      : {run_id}")